In [2]:
require 'json'
require 'rest-client'
require 'linkeddata'
require 'sparql/client'
require 'csv'
require '../Lookups/metadata_functions.rb'

CSVFILE = "./raw_data/bv-kg-20250225.large".freeze
OUTPUT = "./maps/2025-biovista-genes.map".freeze
# puts `ls`
puts `head -2 ./raw_data/bv-kg-20250225.large`

/home/osboxes/CODE/SIMPATHIC2/SKG_Mapping/Lookups/ontologyservers/ncbo.rb:4: warning: already initialized constant PARAMS
/home/osboxes/CODE/SIMPATHIC2/SKG_Mapping/Lookups/ontologyservers/edam.rb:4: warning: previous definition of PARAMS was here


source_1	id_1	type_1	name_1	source_2	id_2	type_2	name_2	score	url
UMLS:Disease or Syndrome:MSH	C0268631	Disease	SSADH Deficiency	HP:human_phenotype	HP:0001263	Human Phenotype	Global developmental delay	0.0501869	https://www.biovista.com/db/link/%5B%5B%22Disease%7CSSADH%20Deficiency%22%5D,%20%5B%22Human%20Phenotype%7CGlobal%20developmental%20delay%22%5D%5D?strength-weight-map=%257B%2522MEDLINE_STRENGTH_AB%2522:1.0,%2522HPO%2522:100.0%257D


In [3]:
ncbi = Hash.new
mesh = Hash.new

# UMLS:Disease or Syndrome:MSH	C0268631	Disease	SSADH Deficiency	
# NCBI:protein-coding	7915	Gene	ALDH5A1	
# 0.166489	https://www.biovista.com/db/link/%5B%5B%22Disease%7CSSADH%20Deficiency%22%5D,%20%5B%22Gene%7CALDH5A1%22%5D%5D?strength-weight-map=%257B%2522MEDLINE_STRENGTH_AB%2522:1.0,%2522HPO%2522:100.0%257D


# UMLS:Disease or Syndrome:MSH	C0268631	Disease	SSADH Deficiency	
# MeSH	D011963	Gene	Receptors, GABA-A	
# 0.000295552	https://www.biovista.com/db/link/%5B%5B%22Disease%7CSSADH%20Deficiency%22%5D,%20%5B%22Gene%7CReceptors,%20GABA-A%22%5D%5D?strength-weight-map=%257B%2522MEDLINE_STRENGTH_AB%2522:1.0,%2522HPO%2522:100.0%257D

# create non-redundant list of genes
# in ncbi{} and mesh{}

CSV.foreach(CSVFILE, headers: true, col_sep: "\t") do |row|
  if row['type_1'] == "Gene"
    if row['source_1'] =~ /NCBI/ || row['source_1'] =~ /UniProt/
      ncbi[row['id_1']] = row['name_1']
    elsif row['source_1'] =~ /MeSH/
      mesh[row['id_1']] = row['name_1']
    else
      abort row['source_1']
    end
  end
  if row['type_2'] == "Gene"
    if row['source_2'] =~ /NCBI/ || row['source_2'] =~ /UniProt/
      ncbi[row['id_2']] = row['name_2']
    elsif row['source_2'] =~ /MeSH/
      mesh[row['id_2']] = row['name_2']
    else
      abort row['source_2']
    end
  end
end
puts "done"


done


# NCBI and UniProt can be dealt with via SPARQL
# MeSH needs an OBO lookup.  Can't get the protein id in that case because it is a class of genes, not a gene...

In [3]:
def format_ncbi_values_clause(idlist:, batch_size: 20)
  # used to make efficient sparql
  valueslist = Array.new
  puts idlist.size
#   slice = 1
  base_uri = "<http://purl.uniprot.org/geneid/%s>"
  idlist.each_slice(batch_size).map do |batch|
#     puts slice
#     slice = slice + 1
    values = batch.map { |id| base_uri % id }.join(' ')
    valueslist << values
  end
  valueslist
end

:format_ncbi_values_clause

In [ ]:
genequery = "
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX uniprotkb: <http://purl.uniprot.org/uniprot/>
PREFIX taxon: <http://purl.uniprot.org/taxonomy/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX up: <http://purl.uniprot.org/core/>
SELECT distinct ?geneid ?protein ?recommended_full ?taxon
WHERE
{
        VALUES ?geneid {|||VALUES|||}
        ?protein a up:Reviewed_Protein .
        ?protein rdfs:seeAlso ?geneid . # seeAlso http://purl.uniprot.org/geneid/939976
        ?protein up:organism ?taxon .
        ?protein up:recommendedName ?rname .
        ?rname up:fullName ?recommended_full .
}"
puts

In [ ]:
out = File.open(OUTPUT, "w");
out.write "bv_geneid,bv_label,geneid,protein,recommended_full,taxon\n"
puts "START"

sparql = SPARQL::Client.new("https://sparql.uniprot.org/sparql/")

batches = format_ncbi_values_clause(idlist: ncbi.keys)
batches.each do |batch|
  retry_attempts = 0
  begin
    result = sparql.query(genequery.gsub("|||VALUES|||", batch))
  rescue StandardError => e
    retry_attempts += 1
    if retry_attempts < 10
      warn "retrying"
      retry
    else
      warn e.inspect
      abort
    end
  end
  puts "FOUND: #{result.size}"
  warn "Batch returned no results — some genes may lack a reviewed UniProt entry" if result.size == 0
  result.each do |res|
    geneuri = res["geneid"].to_s
    if match = geneuri.match(/.*\/([\w\d]+)/)
      geneid = match[1]
      bvlabel = ncbi[geneid]
    else
      abort geneuri
    end
    puts "#{geneid},#{bvlabel},#{geneuri},#{res['protein']},#{res['recommended_full']},#{res['taxon']}"
    out.write CSV.generate_line([geneid,bvlabel,geneuri,res["protein"],res["recommended_full"],res["taxon"]])
  end
end

out.close
puts "done ncbi"

In [4]:
# Patch cell: fix recommended_full in the gene map.
# The SPARQL query may return multiple protein isoforms per bv_geneid, and unreviewed
# TrEMBL entries (A0A... accessions) can sort before the canonical Swiss-Prot entry,
# causing graphing notebooks to pick the wrong label via `find`.
# This patch calls UniProt REST API for every protein accession in the map and
# overwrites recommended_full with the authoritative protein recommended name.
# Run once after regenerating the map; future runs with up:Reviewed_Protein in the
# SPARQL query will return canonical entries only.

require 'net/http'
require 'uri'
require 'json'
require 'csv'

MAP_FILE   = OUTPUT   # './maps/2025-biovista-genes.map'
BATCH_SIZE = 50

rows = CSV.read(MAP_FILE, headers: true).map(&:to_h)
puts "Loaded #{rows.size} rows"

accessions = rows.map { |r| r['protein'].to_s.match(/uniprot\/([A-Z0-9]+)/)&.[](1) }.compact.uniq
puts "#{accessions.size} unique UniProt accessions to fetch"

acc_to_name = {}

accessions.each_slice(BATCH_SIZE).with_index(1) do |batch, i|
  query = batch.map { |a| "accession:#{a}" }.join(' OR ')
  uri   = URI('https://rest.uniprot.org/uniprotkb/search')
  uri.query = URI.encode_www_form(
    query:  query,
    fields: 'accession,protein_name',
    format: 'json',
    size:   BATCH_SIZE + 10
  )

  resp = Net::HTTP.get_response(uri)
  unless resp.is_a?(Net::HTTPSuccess)
    warn "Batch #{i} failed: HTTP #{resp.code}"
    next
  end

  JSON.parse(resp.body)['results'].each do |entry|
    acc  = entry['primaryAccession']
    name = entry.dig('proteinDescription', 'recommendedName', 'fullName', 'value') ||
           entry.dig('proteinDescription', 'submissionNames', 0, 'fullName', 'value')
    acc_to_name[acc] = name if name
  end

  warn "Batch #{i}/#{(accessions.size.to_f / BATCH_SIZE).ceil} done — #{acc_to_name.size} names collected"
  sleep 0.2
end

puts "Fetched #{acc_to_name.size} protein names from UniProt"

no_match = []
rows.each do |row|
  acc = row['protein'].to_s.match(/uniprot\/([A-Z0-9]+)/)&.[](1)
  next unless acc
  if (name = acc_to_name[acc])
    row['recommended_full'] = name
  else
    no_match << acc
  end
end

warn "No UniProt name found for: #{no_match.join(', ')}" unless no_match.empty?

CSV.open(MAP_FILE, 'w', write_headers: true, headers: rows.first.keys) do |csv|
  rows.each { |row| csv << row.values }
end

puts "Done — #{rows.size - no_match.size}/#{rows.size} rows updated in #{MAP_FILE}"

Loaded 958 rows
958 unique UniProt accessions to fetch


Batch 1/20 done — 45 names collected
Batch 2/20 done — 93 names collected
Batch 3/20 done — 143 names collected
Batch 4/20 done — 191 names collected
Batch 5/20 done — 241 names collected
Batch 6/20 done — 291 names collected
Batch 7/20 done — 341 names collected
Batch 8/20 done — 391 names collected
Batch 9/20 done — 440 names collected
Batch 10/20 done — 490 names collected
Batch 11/20 done — 539 names collected
Batch 12/20 done — 586 names collected
Batch 13/20 done — 634 names collected
Batch 14/20 done — 682 names collected
Batch 15/20 done — 731 names collected
Batch 16/20 done — 777 names collected
Batch 17/20 done — 826 names collected
Batch 18/20 done — 875 names collected
Batch 19/20 done — 922 names collected
Batch 20/20 done — 930 names collected


Fetched 930 protein names from UniProt


No UniProt name found for: A6KLE7, A0A9W3SSN0, A0A9X0VWV3, A0A9X5N3X0, B7HJZ1, A6K4M1, A0A6M4JLV7, L7RSL8, A0A6M4JQ33, L7RSM2, A0A182R4I5, W5PLL2, A0A0G2JZF4, A0A0F7RQE8, A0A8D1F8J7, A0A8D1U5L7, A0A1S4D7M9, Q53SB5, A0A182RKJ0, A4D0W4, Q7F6Y3, Q53SX6, A0A6D2XIQ6, A6IIZ6, A9UMW1, A0A1U9X7R0, A6KGI5, A6KGI6


Done — 930/958 rows updated in ./maps/2025-biovista-genes.map


In [ ]:
# Dedup patch: collapse map to one row per bv_geneid, keeping the canonical protein.
# Multiple reviewed UniProt proteins can link to the same NCBI gene ID with different
# recommended names. The canonical entry has an accession starting with O, P, or Q
# (the traditional Swiss-Prot format); other 6-char accessions rank second; TrEMBL
# 10-char accessions (A0A...) rank last.
# Run this after the recommended_full patch cell above.

require 'csv'

MAP_FILE = OUTPUT  # './maps/2025-biovista-genes.map'

rows = CSV.read(MAP_FILE, headers: true).map(&:to_h)
puts "Loaded #{rows.size} rows (#{rows.map { |r| r['bv_geneid'] }.uniq.size} unique genes)"

def accession_rank(protein_uri)
  acc = protein_uri.to_s.match(/uniprot\/([A-Z0-9]+)/)&.[](1) || ''
  return 0 if acc.match?(/^[OPQ]\d[A-Z0-9]{3}\d$/)  # canonical Swiss-Prot
  return 1 if acc.length == 6                          # other reviewed 6-char
  2                                                     # TrEMBL / 10-char
end

best = {}
rows.each do |row|
  id = row['bv_geneid']
  if !best[id] || accession_rank(row['protein']) < accession_rank(best[id]['protein'])
    best[id] = row
  end
end

deduped = best.values
puts "Deduplicated to #{deduped.size} rows (one per gene)"

CSV.open(MAP_FILE, 'w', write_headers: true, headers: rows.first.keys) do |csv|
  deduped.each { |row| csv << row.values }
end

puts "Written to #{MAP_FILE}"

In [10]:
out.close

In [5]:
# For mesh, we will do an OBO lookup on each one

OUTFILE = "./maps/2025-biovista-mesh.map".freeze

out = File.open(OUTFILE, "w")
out.write CSV.generate_line(["bv_geneid"large breasts,,"bv_label","geneid","recommended_full","taxon"])
mesh.each do |m,label|
  uri = "http://purl.bioontology.org/ontology/MESH/#{m}"
  term = ontology_annotations(uri: uri )
  puts "#{uri},#{label},#{term},http://purl.uniprot.org/taxonomy/9606"
  out.write CSV.generate_line([m,label,uri,term,"http://purl.uniprot.org/taxonomy/9606"])
end

puts "done mesh"

SyntaxError: (irb):5: syntax error, unexpected local variable or method, expecting ']'
...generate_line(["bv_geneid"large breasts,,"bv_label","geneid"...
...                          ^~~~~
